## Accelerate Inference: Neural Network Pruning

In [1]:
import os
import numpy as np
import cv2
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# untar
!ls
!tar -xvzf dataset.tar.gz
# load train
train_images = pickle.load(open('train_images.pkl', 'rb'))
train_labels = pickle.load(open('train_labels.pkl', 'rb'))
# load val
val_images = pickle.load(open('val_images.pkl', 'rb'))
val_labels = pickle.load(open('val_labels.pkl', 'rb'))

dataset.tar.gz	train_images.pkl  val_images.pkl
sample_data	train_labels.pkl  val_labels.pkl
train_images.pkl
train_labels.pkl
val_images.pkl
val_labels.pkl


In [4]:
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)

train_images = train_images.permute(0, 3, 1, 2)
val_images = val_images.permute(0, 3, 1, 2)

In [5]:
train_dataset = TensorDataset(train_images,
                              torch.tensor(train_labels.squeeze(), dtype=torch.long))
val_dataset = TensorDataset(val_images,
                            torch.tensor(val_labels.squeeze(), dtype=torch.long))

In [6]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [7]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        self.model = nn.Sequential(
            # First block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Second block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten layer
            nn.Flatten(),

            # Fully connected block: Dense -> ReLU -> Dropout -> Dense -> Softmax
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [8]:
model = ConvNet()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

In [9]:
model = model.to(device)
summary(model, input_size=(3, 25, 25))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 25, 25]             896
              ReLU-2           [-1, 32, 25, 25]               0
            Conv2d-3           [-1, 32, 23, 23]           9,248
              ReLU-4           [-1, 32, 23, 23]               0
         MaxPool2d-5           [-1, 32, 11, 11]               0
           Dropout-6           [-1, 32, 11, 11]               0
            Conv2d-7           [-1, 64, 11, 11]          18,496
              ReLU-8           [-1, 64, 11, 11]               0
            Conv2d-9             [-1, 64, 9, 9]          36,928
             ReLU-10             [-1, 64, 9, 9]               0
        MaxPool2d-11             [-1, 64, 4, 4]               0
          Dropout-12             [-1, 64, 4, 4]               0
          Flatten-13                 [-1, 1024]               0
           Linear-14                  [

In [10]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()  # Zero the parameter gradients
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update tqdm description with current loss and accuracy
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

In [11]:
def validate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the validation loop
    val_loader_tqdm = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():  # Disable gradient calculations for validation
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Update tqdm description with current validation loss and accuracy
            val_loader_tqdm.set_postfix(loss=val_loss / total, accuracy=100 * correct / total)

    val_accuracy = 100 * correct / total
    val_loss = val_loss / len(val_loader)
    return val_loss, val_accuracy

In [12]:
# Main training loop
num_epochs = 50
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    # Training
    train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Validation
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    # Print epoch results
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')

Epoch 1/50


Epoch [1/50], Train Loss: 1.4951, Train Acc: 32.29%, Val Loss: 1.3576, Val Acc: 41.19%
Epoch 2/50


Epoch [2/50], Train Loss: 1.3496, Train Acc: 41.77%, Val Loss: 1.2894, Val Acc: 43.88%
Epoch 3/50


Epoch [3/50], Train Loss: 1.3005, Train Acc: 45.00%, Val Loss: 1.2563, Val Acc: 46.18%
Epoch 4/50


Epoch [4/50], Train Loss: 1.2628, Train Acc: 46.80%, Val Loss: 1.2350, Val Acc: 47.25%
Epoch 5/50


...

Epoch [46/50], Train Loss: 0.7275, Train Acc: 72.37%, Val Loss: 0.7782, Val Acc: 69.35%
Epoch 47/50


Epoch [47/50], Train Loss: 0.7193, Train Acc: 72.51%, Val Loss: 0.7810, Val Acc: 69.47%
Epoch 48/50


Epoch [48/50], Train Loss: 0.7085, Train Acc: 72.69%, Val Loss: 0.7850, Val Acc: 68.79%
Epoch 49/50


Epoch [49/50], Train Loss: 0.7067, Train Acc: 73.18%, Val Loss: 0.7850, Val Acc: 69.47%
Epoch 50/50


Epoch [50/50], Train Loss: 0.7005, Train Acc: 73.65%, Val Loss: 0.7950, Val Acc: 68.44%


In [13]:
torch.save(model.state_dict(), 'my_model_weights_1.pt', _use_new_zipfile_serialization=False)

### L1-Norm Filter Pruning

This method is **L1-Norm Filter Pruning**, where filters are ranked by their L1-norm and the least important ones are zeroed out.

We calculate the L1-norm (sum of absolute weights) for each filter.

Units with the lowest L1-norms are masked (set to zero) to achieve the target sparsity.

In [14]:
def compute_l1_mask(model, target_sparsity, device):
    mask = {}
    # Layers to exclude from pruning (Protect Input and Output)
    excluded_layers = ["model.16", "model.0"]
    with torch.no_grad():
        for name, param in model.named_parameters():
            if "weight" in name:
                if any(ex in name for ex in excluded_layers):
                    continue
                # Calculate L1 norms for this specific layer
                if param.dim() == 4:
                    norms = param.abs().sum(dim=(1, 2, 3))
                    view_shape = (-1, 1, 1, 1)
                elif param.dim() == 2:
                    norms = param.abs().sum(dim=1)
                    view_shape = (-1, 1)
                else:
                    continue

                # Layer-wise Thresholding
                # This prevents smaller layers (like Conv2) from being wiped out by larger layers
                threshold = torch.quantile(norms, target_sparsity)
                layer_mask = (norms > threshold).float().to(device)
                layer_mask = layer_mask.view(*view_shape)

                mask[name] = layer_mask
                param.mul_(layer_mask)

    return mask

def train_one_epoch_masked(model, train_loader, optimizer, criterion, device, mask=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        if mask is not None:
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in mask and param.grad is not None:
                        param.grad.mul_(mask[name])

        optimizer.step()

        if mask is not None:
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in mask:
                        param.mul_(mask[name])

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    return running_loss / len(train_loader), 100 * correct / total

def compute_sparsity(model):
    total_zeros = sum(torch.sum(p == 0).item() for p in model.parameters())
    total_params = sum(p.numel() for p in model.parameters())
    return total_zeros / total_params

In [15]:
# val_loss, val_accuracy = validate(model, val_loader, criterion, device)

In [16]:
# val_accuracy

In [17]:
configurations = [
    {'num_epochs': 20, 'final_sparsity': 0.60, 'steps': 5},
    {'num_epochs': 25, 'final_sparsity': 0.65, 'steps': 5},
    {'num_epochs': 30, 'final_sparsity': 0.70, 'steps': 8},
    {'num_epochs': 35, 'final_sparsity': 0.75, 'steps': 8},
]

best_score = 0
best_config = None
best_state = None

for config in configurations:
    print(f"\n{'='*70}\nTesting L1 Config: {config}\n{'='*70}")

    model.load_state_dict(torch.load('my_model_weights_1.pt'))
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)
    mask = None

    total_epochs = config['num_epochs']
    steps = config['steps']
    epochs_per_step = total_epochs // (steps + 1)
    current_iter = 0

    for epoch in range(total_epochs):
        is_pruning_epoch = (epoch % epochs_per_step == 0) and (current_iter < steps)

        if is_pruning_epoch:
            current_iter += 1
            target_sparsity = (config['final_sparsity'] / steps) * current_iter
            mask = compute_l1_mask(model, target_sparsity, device)
            print(f"  [Step {current_iter}/{steps}] Target: {target_sparsity:.4f}, Actual: {compute_sparsity(model):.4f}")

        train_loss, train_acc = train_one_epoch_masked(
            model, train_loader, optimizer, criterion, device, mask
        )

        if (epoch + 1) % 5 == 0 or epoch == total_epochs - 1:
            val_loss, val_acc = validate(model, val_loader, criterion, device)
            current_sparsity = compute_sparsity(model)
            score = (val_acc/100 + current_sparsity) / 2 if val_acc > 60 else 0

            print(f"  Epoch {epoch+1}: Val Acc: {val_acc:.2f}%, Sparsity: {current_sparsity:.4f}, Score: {score:.4f}")

            if score > best_score:
                best_score = score
                best_config = config
                best_state = model.state_dict().copy()
                print(f" New Best Score: {best_score:.4f}")

print(f"\n{'='*70}\nBest Config: {best_config}\nBest Score: {best_score:.4f}\n{'='*70}")

if best_state is not None:
    torch.save(best_state, 'my_model_weights_L1.pt', _use_new_zipfile_serialization=False)
    print("\nBest model saved as 'my_model_weights_L1.pt'")


Testing L1 Config: {'num_epochs': 20, 'final_sparsity': 0.6, 'steps': 5}
  [Step 1/5] Target: 0.1200, Actual: 0.1207


  [Step 2/5] Target: 0.2400, Actual: 0.2396


  Epoch 5: Val Acc: 68.48%, Sparsity: 0.2396, Score: 0.4622
 New Best Score: 0.4622


  [Step 3/5] Target: 0.3600, Actual: 0.3571


  [Step 4/5] Target: 0.4800, Actual: 0.4773


...

  Epoch 20: Val Acc: 62.14%, Sparsity: 0.6517, Score: 0.6365


  [Step 8/8] Target: 0.7500, Actual: 0.7448


  Epoch 25: Val Acc: 56.48%, Sparsity: 0.7448, Score: 0.0000


  Epoch 30: Val Acc: 61.78%, Sparsity: 0.7448, Score: 0.6813
 New Best Score: 0.6813


  Epoch 35: Val Acc: 62.34%, Sparsity: 0.7448, Score: 0.6841
 New Best Score: 0.6841

Best Config: {'num_epochs': 35, 'final_sparsity': 0.75, 'steps': 8}
Best Score: 0.6841

Best model saved as 'my_model_weights_L1.pt'
